<a href="https://colab.research.google.com/github/shaheem-anwar/DLBDSMLUS01-mental-health-clustering-MohammedShaheemAnwar/blob/main/Cartpole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## First installing the library of the gym 🫠

In [2]:
!pip install gymnasium stable-baselines3

# Now I'm starting the codes: 💻

In [3]:
# Importing Libraries
import gymnasium as gym
from stable_baselines3 import PPO

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 🚀 Making it Better: Centered CartPole Environment
We can create a custom Reward Wrapper to penalize the agent if it drifts away from the center ($x = 0$). This forces the agent to learn to balance the pole specifically in the middle of the screen.

In [14]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO

class CenteredRewardWrapper(gym.RewardWrapper):
    def __init__(self, env):
        super().__init__(env)

    def reward(self, reward):
        # Get current state observation
        # obs[0] is the cart position (x), which ranges from -2.4 to 2.4
        obs = self.env.unwrapped.state
        cart_position = obs[0]

        # Penalize the agent proportionally to its distance squared from the center
        # If it stays at 0, no penalty. If it drifts to 2.4, it gets a heavy penalty.
        penalty = 0.5 * (cart_position ** 2)
        return reward - penalty

# Create the environment with our custom reward shape
base_env = gym.make("CartPole-v1")
env = CenteredRewardWrapper(base_env)

# train the model with the improved reward structure
rl_model = PPO("MlpPolicy", env, verbose=1)
rl_model.learn(total_timesteps=100000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 18.9     |
|    ep_rew_mean     | 18.9     |
| time/              |          |
|    fps             | 1369     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 27.4         |
|    ep_rew_mean          | 26.7         |
| time/                   |              |
|    fps                  | 986          |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0083780745 |
|    clip_fraction        | 0.0843       |
|    clip_range           | 0.2          |
|    en

In [17]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from base64 import b64encode
import os
from stable_baselines3 import PPO

# Create a trigger function to record all episodes
def record_trigger(episode_id):
    return True

# Initialize environment for video recording
trigger_env = gym.make("CartPole-v1", render_mode="rgb_array")

# Wrap environment to save the video
video_env = RecordVideo(trigger_env, video_folder="./video", episode_trigger=record_trigger, disable_logger=True)

# Let's run for 3 full episodes to get a solid, non-empty video
for episode in range(3):
    obs, _ = video_env.reset()
    done = False
    while not done:
        action, _ = rl_model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = video_env.step(action)
        done = terminated or truncated

# Explicitly close the environment to flush and finalize the video files
video_env.close()
trigger_env.close()

# Helper function to find and display the recorded mp4 file
video_files = [f for f in os.listdir('./video') if f.endswith('.mp4')]
if video_files:
    # Read the latest recorded video file
    latest_video = sorted(video_files)[-1]
    print(f"Displaying latest video: {latest_video}")
    mp4 = open(f'./video/{latest_video}', 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=600 controls autoplay loop>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("No video recorded.")

/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:298: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Displaying latest video: rl-video-episode-2.mp4


In [16]:
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym

# Create a clean evaluation environment
eval_env = gym.make("CartPole-v1")

# Evaluate the trained policy over 10 episodes
mean_reward, std_reward = evaluate_policy(rl_model, eval_env, n_eval_episodes=10, deterministic=True)

print(f"Evaluation results over 10 episodes:")
print(f"Mean Reward: {mean_reward:.2f}")
print(f"Standard Deviation of Reward: {std_reward:.2f}")

# A perfect CartPole score over an entire episode is 500.0
if mean_reward >= 450.0:
    print("Excellent performance! The model is keeping the pole balanced successfully.")
else:
    print("The model could still be improved.")

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Evaluation results over 10 episodes:
Mean Reward: 500.00
Standard Deviation of Reward: 0.00
Excellent performance! The model is keeping the pole balanced successfully.
